In [6]:
import duckdb
import pandas as pd
import numpy as np
from pathlib import Path

DB_PATH = "developer_project.duckdb"
EXPORT_DIR = Path("toexport")
EXPORT_DIR.mkdir(exist_ok=True)

con = duckdb.connect(DB_PATH)

pd.set_option("display.max_columns", 250)
pd.set_option("display.max_rows", 100)

ACTIVITY_TABLE = "activity_final"
CONTACT_TABLE = "contact_final"

# Robust helper: use natural log of 1+x in DuckDB.
LOG1P = "LN(1 + {x})"

EFFORT_MAPPING_PATH = Path("Data/Activity_Score_Mapping_filled.xlsx")
EFFORT_MAPPING_SHEET = "Activity_Score_Mapping"

table_names = [
    "activity_base_v2",
    "activity_dictionary_v2",
    "activity_effort_mapping_ai_v2",
    "activity_labeled_v2",
    "contact_one_row_v2",
    "dev_activation_v2",
    "dev_contact_persona_v2",
    "dev_dormancy_status_v2",
    "dev_effort_level_v2",
    "dev_features_lifetime_v2",
    "dev_journey_state_v2",
    "dev_meaningful_week_v2",
    "dev_persona_v2",
    "dev_profile_final_v4",
    "dev_recency_features_v2",
    "dev_weekly_features_v2",
    "developer_universe_v2"
]

# Show tables
print(con.execute("SHOW TABLES").fetch_df())

# Export all tables to parquet
for table in table_names:
    try:
        output_path = EXPORT_DIR / f"{table}.parquet"

        con.execute(f"""
        COPY {table}
        TO '{output_path.as_posix()}'
        (FORMAT PARQUET)
        """)

        print(f"Exported: {output_path}")

    except Exception as e:
        print(f"Failed exporting {table}: {e}")

con.close()

                                          name
0                             activity_base_v2
1                       activity_dictionary_v2
2                activity_effort_mapping_ai_v2
3                               activity_final
4                          activity_labeled_v2
..                                         ...
105  developer_clusters_v3_corr_groups_dormant
106              developer_clusters_v3_dormant
107                      developer_universe_v2
108                         sdk_download_final
109                           sdk_download_raw

[110 rows x 1 columns]
100% ▕██████████████████████████████████████▏ (00:00:04.82 elapsed)     
Exported: toexport/activity_base_v2.parquet
Exported: toexport/activity_dictionary_v2.parquet
Exported: toexport/activity_effort_mapping_ai_v2.parquet
100% ▕██████████████████████████████████████▏ (00:00:34.51 elapsed)     
Exported: toexport/activity_labeled_v2.parquet
Exported: toexport/contact_one_row_v2.parquet
Exported: toexport/dev

In [5]:
import duckdb
import pandas as pd
from pathlib import Path

DB_PATH = "developer_project.duckdb"
EXPORT_DIR = Path("toexport_clusters")
EXPORT_DIR.mkdir(exist_ok=True)

con = duckdb.connect(DB_PATH)

pd.set_option("display.max_columns", 250)
pd.set_option("display.max_rows", 100)

# Cluster output tables from clustering_v11_hdbscan_main_rule_dormant_skip_existing.ipynb
cluster_table_names = [
    "dev_lifecycle_cluster_membership_v11_active",
    "dev_lifecycle_cluster_membership_v11_cooling",
    "dev_lifecycle_cluster_membership_v11_at_risk",
    "dev_lifecycle_cluster_membership_v11_dormant",
    "dev_lifecycle_cluster_membership_v11_unactivated",
    "dev_lifecycle_cluster_membership_v11_combined",
    "dev_lifecycle_cluster_membership_v11_final",
    "dev_lifecycle_cluster_membership_v11_profile_summary",
    "dev_lifecycle_cluster_run_stats_v11",
]

# Optional older/reference cluster tables if you want to export comparison runs too
optional_cluster_tables = [
    "developer_clusters_v1",
    "developer_clusters_v2",
    "developer_clusters_v3",
    "developer_clusters_v3_active",
    "developer_clusters_v3_cooling",
    "developer_clusters_v3_dormant",
    "dev_lifecycle_cluster_membership_v10_active",
    "dev_lifecycle_cluster_membership_v11_profile_summary",
]

# Get existing tables from DuckDB
existing_tables = set(con.execute("SHOW TABLES").fetch_df().iloc[:, 0].astype(str))

print("Cluster-related tables found:")
cluster_like = sorted([t for t in existing_tables if "cluster" in t.lower()])
for t in cluster_like:
    print("-", t)

# Export only tables that actually exist
tables_to_export = []
for table in cluster_table_names + optional_cluster_tables:
    if table in existing_tables and table not in tables_to_export:
        tables_to_export.append(table)

print("\nTables selected for export:")
for table in tables_to_export:
    print("-", table)

# Export selected cluster tables to parquet
for table in tables_to_export:
    try:
        output_path = EXPORT_DIR / f"{table}.parquet"

        con.execute(f"""
        COPY {table}
        TO '{output_path.as_posix()}'
        (FORMAT PARQUET)
        """)

        print(f"Exported: {output_path}")

    except Exception as e:
        print(f"Failed exporting {table}: {e}")

con.close()
print("\nDone.")

Cluster-related tables found:
- cluster_comparison_metrics_v1
- cluster_label_map_feature_v1
- cluster_label_map_v1
- cluster_noise_sweep_v1
- clustering_cohort_dormancy_active_v1
- clustering_cohort_dormancy_cooling_v1
- clustering_cohort_dormancy_dormant_v1
- clustering_cohort_v1
- clustering_cohort_v2_active
- clustering_cohort_v2_cooling
- clustering_cohort_v2_dormant
- clustering_cohort_v3_active
- clustering_cohort_v3_cooling
- clustering_cohort_v3_dormant
- clustering_cohort_v3_persona_active_cuda
- clustering_cohort_v3_persona_active_genai
- clustering_cohort_v3_persona_active_learning_community
- clustering_cohort_v3_persona_active_robotics
- clustering_cohort_v3_persona_active_simulation
- clustering_cohort_v3_persona_active_unknown_or_other
- clustering_cohort_v3_persona_cooling_cuda
- clustering_cohort_v3_persona_cooling_genai
- clustering_cohort_v3_persona_cooling_learning_community
- clustering_cohort_v3_persona_cooling_robotics
- clustering_cohort_v3_persona_cooling_simu

In [1]:
import duckdb
import pandas as pd
from pathlib import Path

DB_PATH = "developer_project.duckdb"
EXPORT_DIR = Path("toexport_clusters")
EXPORT_DIR.mkdir(exist_ok=True)

con = duckdb.connect(DB_PATH)

gmm_tables = [
    "dev_gmm_stratified_clusters_v1",
    "dev_gmm_weekly_clusters_v1",
]

existing_tables = set(con.execute("SHOW TABLES").fetchdf().iloc[:, 0].astype(str))

for table in gmm_tables:
    if table not in existing_tables:
        print(f"Skipping missing table: {table}")
        continue

    output_path = EXPORT_DIR / f"{table}.parquet"

    con.execute(f"""
        COPY {table}
        TO '{output_path.as_posix()}'
        (FORMAT PARQUET)
    """)

    print(f"Exported: {output_path}")

con.close()

Exported: toexport_clusters/dev_gmm_stratified_clusters_v1.parquet
Exported: toexport_clusters/dev_gmm_weekly_clusters_v1.parquet
